In [ ]:
!git clone https://github.com/trextrader/hotdogornot

In [ ]:
cd hotdogornot/

In [ ]:
cd training

In [ ]:
# @title
!pip install -e ".[dev]"

In [ ]:
!git pull

In [ ]:
%cd /content/hotdogornot
!ls training/data/labeled/embedder
!python -c "import sys; sys.path.insert(0,'training'); from rfconnectorai.data.classes import class_names; print('classes:', class_names('training/configs/classes.yaml'))"

In [ ]:
%%shell
  set -euxo pipefail
  cd /content/hotdogornot
  export PYTHONPATH=/content/hotdogornot/training
  export PYTHONUNBUFFERED=1

  python -u -m rfconnectorai.data.audit \
      --data-dir training --out docs/DATASET_AUDIT.md

  echo "----- audit summary -----"
  head -40 docs/DATASET_AUDIT.md

  python -u -m rfconnectorai.data.crop_instances \
      --input training/data/labeled/embedder \
      --manifest datasets/rfconnectors/instances.jsonl \
      --out datasets/rfconnectors/crops \
      --mode whole-image --base-dir training

  wc -l datasets/rfconnectors/instances.jsonl

In [ ]:
%%shell
  set -euxo pipefail
  cd /content/hotdogornot
  export PYTHONPATH=/content/hotdogornot/training
  export PYTHONUNBUFFERED=1

  python -u -m rfconnectorai.data.build_yolo_dataset \
      --input datasets/rfconnectors/instances.jsonl \
      --out datasets/rfconnectors --base-dir training \
      --single-class \
      --taxonomy training/rfconnectorai/specs/connectors.yaml

  cat datasets/rfconnectors/data.yaml   # expect nc: 1, names: [connector]

In [ ]:
%%shell
# Stage 1: single-class connector localizer (data.yaml is now nc=1).
set -euxo pipefail
cd /content/hotdogornot
export PYTHONPATH=/content/hotdogornot/training
export PYTHONUNBUFFERED=1

python -u -m rfconnectorai.detector.train_yolo \
    --data datasets/rfconnectors/data.yaml --model yolo11n.pt \
    --epochs 10 --imgsz 640 --batch 96 --device 0 \
    --dataset-lock datasets/rfconnectors/dataset.lock.json \
    --out reports/experiments/detector_run_full \
    --artifact-out models/detector


In [ ]:
%%shell
set -euxo pipefail
cd /content/hotdogornot
export PYTHONPATH=/content/hotdogornot/training
export PYTHONUNBUFFERED=1

# CURATED-ONLY (2026-05-16). The 196 real field photos at repo-root
# data/labeled/embedder cover all 9 classes incl. 1.85mm M/F and SMA-F.
# The legacy training/data/labeled/embedder set is synthetic and MISSING
# 1.85mm-M, 1.85mm-F, SMA-F, so it is intentionally NOT used. Single
# classifier run; output models/connector_classifier feeds eval+export.
# Hyperparams below are exactly what produced the committed model
# (epochs raised 30->50 for the production run; lr 1e-4).
python -u -m rfconnectorai.classifier.train \
    --data-dir data/labeled/embedder \
    --out-dir models/connector_classifier \
    --architecture efficientnet_v2_s --input-size 384 \
    --epochs 50 --batch-size 16 --lr 1e-4

In [ ]:
# Phase 2 staged fine-tune is intentionally SKIPPED under the curated-only
# strategy (2026-05-16). The legacy corpus is synthetic and missing 3 of
# the 9 classes, so there is no separate Phase-1 corpus to warm-start
# from. Cell 9 above is the single curated training run; its output
# models/connector_classifier is consumed by the eval + export cells.
print("Phase 2 skipped: curated-only training (see the cell above).")

In [ ]:
%%shell
set -euo pipefail
cd /content/hotdogornot
export PYTHONPATH=/content/hotdogornot/training
python - <<'PY'
import json, torch
from pathlib import Path
from rfconnectorai.classifier.train import build_model
from rfconnectorai.classifier.dataset import ConnectorFolderDataset, make_eval_transforms
from rfconnectorai.data.classes import class_names
from rfconnectorai.eval.nine_class_report import build_report, render_markdown

cn = class_names("training/configs/classes.yaml")
md = Path("models/connector_classifier")
lab = json.loads((md / "labels.json").read_text())
arch = lab.get("architecture", "efficientnet_v2_s")
size = lab.get("input_size", 384)

m = build_model(len(cn), arch)
m.load_state_dict(torch.load(md / "weights.pt", map_location="cpu"))
m.eval()

ds = ConnectorFolderDataset("data/labeled/embedder", cn,
                            transform=make_eval_transforms(size))
yt, yp = [], []
with torch.no_grad():
    for i in range(len(ds)):
        x, y = ds[i]
        yt.append(y)
        yp.append(int(m(x.unsqueeze(0)).argmax(1)))

rep = build_report(yt, yp, cn)
out = Path("reports/experiments/classifier_9class")
out.mkdir(parents=True, exist_ok=True)
(out / "REPORT.md").write_text(render_markdown(rep))
print(render_markdown(rep))
print("NOTE: scored over ALL 196 imgs incl. training data -> headline "
      "accuracy is optimistic; read the CONFUSION STRUCTURE (which "
      "classes bleed into which, M<->F, adjacent sizes), not the number.")
PY

In [ ]:
# Export ONNX, drop bump_version's byte-identical copies, zip (RELATIVE
# paths so extraction never nests content/hotdogornot or spawns dups),
# then download. ~160MB (1 .pt + 1 .onnx) instead of ~560MB.
!python -m rfconnectorai.classifier.export_onnx     --model-dir /content/hotdogornot/models/connector_classifier     --output /content/hotdogornot/models/connector_classifier/classifier.onnx

!cd /content/hotdogornot/models/connector_classifier && rm -f     weights.0001.pt weights.latest.pt     weights.0001.onnx weights.latest.onnx weights.onnx

!mkdir -p /content/hotdogornot/reports/experiments/classifier_9class
!cp /content/hotdogornot/models/connector_classifier/metrics.json     /content/hotdogornot/models/connector_classifier/version.json     /content/hotdogornot/models/connector_classifier/labels.json     /content/hotdogornot/reports/experiments/classifier_9class/

!rm -f /content/classifier_9class.zip
!cd /content/hotdogornot && zip -r /content/classifier_9class.zip     models/connector_classifier reports/experiments/classifier_9class

from google.colab import files
files.download('/content/classifier_9class.zip')


In [ ]:
# Detector run: zip with RELATIVE paths (no /content/hotdogornot nesting)
# and exclude weights/last.pt (byte-identical to best.pt).
!rm -f /content/detector_run_full.zip
!cd /content/hotdogornot && zip -r /content/detector_run_full.zip     reports/experiments/detector_run_full models/detector/best.pt     -x '*/weights/last.pt'

from google.colab import files
files.download('/content/detector_run_full.zip')


In [ ]:
# Check which runs exist and their sizes
!ls -la /content/hotdogornot/reports/experiments/
!wc -l /content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/results.csv
!cat /content/hotdogornot/reports/experiments/detector_run_full/ultralytics/connector/results.csv
